# Measurement: what you actually see
**Quantum Computing · Week 1 (Qubits and measurement) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qc/w1d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A measurement gives one random bit and collapses the qubit.
Many shots on fresh qubits turn bits into probabilities.

In [ ]:
import numpy as np
from qiskit.quantum_info import Statevector

plus = Statevector([1, 1]) / np.sqrt(2)
plus.seed(3)                                   # fixed seed, so the "random" outcome repeats
outcome, after = plus.measure()               # one measurement: an outcome and the state afterwards
print("first measurement: ", outcome, "  state afterwards:", np.round(after.data, 3))
outcome2, _ = after.measure()
print("second measurement:", outcome2, " (always the same as the first)")

## Many shots with a sampler
A circuit prepares a state with $P(1) = 0.3$ (the `ry` gate turns the qubit; you'll meet it properly
in week 2) and measures it. The sampler runs it for as many shots as we ask.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

theta = 2 * np.arcsin(np.sqrt(0.3))           # the turn that gives P(1) = 0.3
qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()

sampler = StatevectorSampler(seed=11)
for shots in [10, 100, 1000, 10000]:
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    est = counts.get("1", 0) / shots
    err = np.sqrt(est * (1 - est) / shots)
    print(f"{shots:6d} shots: {counts}  ->  P(1) = {est:.3f} ± {err:.3f}")

## Is the error formula right?
Repeat a 1000-shot experiment 300 times. The estimates should spread by about
$\sqrt{0.3 \cdot 0.7 / 1000} \approx 0.0145$.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
estimates = rng.binomial(1000, 0.3, size=300) / 1000     # 300 experiments of 1000 shots each
predicted = np.sqrt(0.3 * 0.7 / 1000)
print(f"spread of the 300 estimates: {estimates.std():.4f}   formula: {predicted:.4f}")

plt.figure(figsize=(6.2, 3.2))
plt.hist(estimates, bins=20, color="#b9a7ff", edgecolor="white")
plt.axvline(0.3, color="#ff7a59", lw=2, label="true P(1) = 0.3")
plt.axvspan(0.3 - 2 * predicted, 0.3 + 2 * predicted, color="#ff7a59", alpha=0.12, label="± 2 typical errors")
plt.xlabel("estimate of P(1) from 1000 shots")
plt.ylabel("experiments")
plt.legend()
plt.show()

## Your turn: plan a shot budget
How many shots give a typical error of ± 0.01 when $p \approx 0.5$? Compute it, then check it with 300 experiments.

In [ ]:
p = 0.5
target = 0.01
N = 100   # TODO: solve target = sqrt(p * (1 - p) / N) for N

estimates = rng.binomial(N, p, size=300) / N
print(f"N = {N}: spread of 300 estimates = {estimates.std():.4f}  (target {target})")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
N = int(np.ceil(p * (1 - p) / target ** 2))       # = 0.25 / 0.0001 = 2500
estimates = rng.binomial(N, p, size=300) / N
print(f"N = {N}: spread of 300 estimates = {estimates.std():.4f}  (target {target})")
print("For ± 0.001 you would need 250,000 shots: precision gets expensive fast.")